# Question 1

Import data from CSV

In [1]:
import pandas as pd
import numpy as np
import csv
import warnings
warnings.filterwarnings("ignore")

raw_data = pd.read_csv('./data.tsv', sep='\t', quoting=csv.QUOTE_NONE)

Prepare Data, by sampling and adding the label

In [2]:
def label(rating):
    if rating > 3:
        return  1
    if rating < 3:
        return 2
    if rating == 3:
        return 3

sampled = raw_data.groupby("star_rating").sample(n=50000, random_state=42)
del raw_data

dataset = pd.DataFrame()
dataset["review"] = sampled["review_body"]
dataset["star_rating"] = sampled["star_rating"]
dataset["sentiment"] = dataset["star_rating"].apply(label)

del sampled


We will perform train / test split after extracting the features from the sentences

# Question 2(a)

Import gensim and load pre-trained model

In [3]:
import gensim.downloader as api
wv = api.load('word2vec-google-news-300')

Test word embeddings and semantics

In [4]:
# Example comparing king, man and woman. Expecting to see queen
wx = wv['king'] - wv['man'] + wv['woman']
wv.most_similar(wx, topn=5)

[('king', 0.8449392318725586),
 ('queen', 0.7300517559051514),
 ('monarch', 0.645466148853302),
 ('princess', 0.6156251430511475),
 ('crown_prince', 0.5818676352500916)]

In [5]:
# Example comparing boy, man and puppy. Expecting to see dog
wx = wv['man'] - wv['boy'] + wv['puppy']
wv.most_similar(wx, topn=5)

[('puppy', 0.7963186502456665),
 ('dog', 0.730284571647644),
 ('pooch', 0.6746339201927185),
 ('puppies', 0.6635603904724121),
 ('cat', 0.659332275390625)]

# Question 2(b)

Create model from reviews

In [4]:
from gensim import utils
import gensim.models

class Corpus:
    def __iter__(self):
        for sentence in dataset["review"].tolist():
            yield utils.simple_preprocess(str(sentence))

sentences = Corpus()
model = gensim.models.Word2Vec(sentences=sentences, vector_size=300, window=11)

Retry examples from previous part

In [ ]:
wx = model.wv["king"] - model.wv["man"] + model.wv["woman"]
model.wv.most_similar(wx, topn=5)

[('subject', 0.4174935221672058),
 ('hybrid', 0.3757959008216858),
 ('notations', 0.3707139790058136),
 ('spiral', 0.36345916986465454),
 ('subjects', 0.3612690269947052)]

In [8]:
wx = model.wv['man'] - model.wv['boy'] + model.wv['puppy']
model.wv.most_similar(wx, topn=5)

[('man', 0.6538234949111938),
 ('woman', 0.47778603434562683),
 ('lady', 0.43342334032058716),
 ('helping', 0.40876027941703796),
 ('chatting', 0.4072163701057434)]

# Question 3

Preprocess sentences and create embeddings

In [7]:
import re
from bs4 import BeautifulSoup
import contractions
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [33]:
def preprocess_and_vectorize(wv):
    def f(review):
        text = str(review).lower()
        text = BeautifulSoup(text, "html.parser").get_text(strip=True)
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = contractions.fix(text)
        stop = set(stopwords.words('english'))
        words = [lemmatizer.lemmatize(w) for w in word_tokenize(text) if w not in stop]
        vectors = [wv[w] if w in wv else [0]*300 for w in words]

        if(len(vectors) == 0): return [0]*300
        else: return np.mean(np.array(vectors), axis=0)
    
    return f;

In [ ]:
dataset["feature_pretrained"] = dataset["review"].apply(preprocess_and_vectorize(wv))
dataset["feature_custom"] = dataset["review"].apply(preprocess_and_vectorize(model.wv))

Train Test split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
binary_ds = dataset[dataset["sentiment"] != 3]
binary_ds["sentiment"] = binary_ds["sentiment"].apply(lambda x: 1 if x==1 else 0)
X_train, X_test, y_train, y_test = train_test_split(binary_ds[["feature_pretrained", "feature_custom"]], binary_ds["sentiment"], test_size=0.2, random_state=42)

General testing skeleton

In [ ]:
from sklearn.metrics import accuracy_score

prep_input = lambda x: pd.DataFrame(x.tolist(), index=x.index)

def test_model(model, feature, name):
    model.fit(prep_input(X_train[feature]), y_train)
    y_pred = model.predict(prep_input(X_test[feature]))
    print(f"{name} accuracy: {accuracy_score(y_test, y_pred)}")

In [14]:
from sklearn.linear_model import Perceptron
test_model(Perceptron(random_state=42), "feature_pretrained", "Perceptron with 'word2vec-google-news-300' features")
test_model(Perceptron(random_state=42), "feature_custom", "Perceptron with self trained Word2Vec features")

Perceptron with 'word2vec-google-news-300' features accuracy: 0.73535
Perceptron with self trained Word2Vec features accuracy: 0.7232


In [17]:
from sklearn.svm import LinearSVC
test_model(LinearSVC(random_state=42), "feature_pretrained", "SVM with 'word2vec-google-news-300' features")
test_model(LinearSVC(random_state=42), "feature_custom", "SVM with self trained Word2Vec features")

SVM with 'word2vec-google-news-300' features accuracy: 0.812025
SVM with self trained Word2Vec features accuracy: 0.835325


# Question 4(a)

Prepare inputs

In [15]:
import keras
from keras.models import Sequential
from keras.layers import Dense

def test_fnn(input_shape, feature_name, is_ternary, name):
    initializer = keras.initializers.HeNormal(seed=42)

    output_size = 3 if is_ternary else 2

    model = Sequential([
        Dense(50, input_shape=input_shape, kernel_initializer=initializer, activation='relu'),
        Dense(10, activation='relu', kernel_initializer=initializer),
        Dense(output_size, activation='softmax', kernel_initializer=initializer)
    ])

    model.compile(optimizer=keras.optimizers.Adam(0.001),
                loss='categorical_crossentropy' if is_ternary else 'binary_crossentropy',
                metrics=['accuracy'])
    
    model.fit(prep_input(X_train[feature_name]), pd.get_dummies(y_train), epochs=10, batch_size=100, verbose=0)

    score = model.evaluate(prep_input(X_test[feature_name]), pd.get_dummies(y_test), verbose=0)

    print(f"{name} accuracy: {score[1]}")


2026-02-27 14:47:19.131536: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-27 14:47:19.982569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-27 14:47:22.710096: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [20]:
test_fnn((300,), 'feature_pretrained', False, "Binary, Google")

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-27 12:46:48.315836: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Binary, Google accuracy: 0.8407750129699707


In [21]:
test_fnn((300,), 'feature_custom', False, "Binary, Self")

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Binary, Self accuracy: 0.8591499924659729


In [22]:
X_train, X_test, y_train, y_test = train_test_split(dataset[["feature_pretrained", "feature_custom"]], dataset["sentiment"], test_size=0.2, random_state=42)

In [23]:
test_fnn((300,), 'feature_pretrained', True, "Ternary, Google")

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Ternary, Google accuracy: 0.6812599897384644


In [24]:
test_fnn((300,), 'feature_custom', True, "Ternary, Self")

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Ternary, Self accuracy: 0.6966999769210815


# Question 4(b)

Prepare dataset by concatenating the first 10 vectors of words

In [8]:
def preprocess_and_vectorize2(wv):
    def f(review):
        text = str(review).lower()
        text = BeautifulSoup(text, "html.parser").get_text(strip=True)
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = contractions.fix(text)
        stop = set(stopwords.words('english'))
        words = [lemmatizer.lemmatize(w) for w in word_tokenize(text) if w not in stop]
        vectors = [wv[w] if w in wv else [0]*300 for w in words]

        if(len(vectors) == 0): return [0]*3000
        else: return np.concatenate(vectors[:10])
    
    return f;

In [9]:
dataset = dataset.groupby("star_rating").sample(n=5000, random_state=42)

In [ ]:
dataset2 = pd.DataFrame()
dataset2['feature_pretrained'] = dataset['review'].apply(preprocess_and_vectorize2(wv))
dataset2['feature_custom'] = dataset['review'].apply(preprocess_and_vectorize2(model.wv))
dataset2['sentiment'] = dataset['sentiment']

In [13]:
def pad(vec):
    if len(vec) < 3000:
        return np.concat([vec, np.zeros(3000-len(vec))])
    else:
        return vec

dataset2['feature_pretrained'] = dataset2['feature_pretrained'].apply(pad)
dataset2['feature_custom'] = dataset2['feature_custom'].apply(pad)

Perform classification with FFN

In [14]:
binary_ds = dataset2[dataset2["sentiment"] != 3]
binary_ds["sentiment"] = binary_ds["sentiment"].apply(lambda x: 1 if x==1 else 0)
X_train, X_test, y_train, y_test = train_test_split(binary_ds[["feature_pretrained", "feature_custom"]], binary_ds["sentiment"], test_size=0.2, random_state=42)

In [18]:
test_fnn((3000,), "feature_pretrained", False, "Binary, Google")

2026-02-27 13:53:23.162262: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-02-27 13:53:33.079400: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 192000000 exceeds 10% of free system memory.


Binary, Google accuracy: 0.7212499976158142


In [19]:
test_fnn((3000,), "feature_custom", False, "Binary, Self")

2026-02-27 13:54:56.107074: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 192000000 exceeds 10% of free system memory.


Binary, Self accuracy: 0.7300000190734863


In [22]:
X_train, X_test, y_train, y_test = train_test_split(dataset2[["feature_pretrained", "feature_custom"]], dataset2["sentiment"], test_size=0.2, random_state=42)

In [24]:
test_fnn((3000,), "feature_pretrained", True, "Ternary, Google")

2026-02-27 14:02:23.399818: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 240000000 exceeds 10% of free system memory.


Ternary, Google accuracy: 0.5644000172615051


In [25]:
test_fnn((3000,), "feature_custom", True, "Ternary, Self")

2026-02-27 14:02:48.767626: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 240000000 exceeds 10% of free system memory.


Ternary, Self accuracy: 0.5526000261306763


# Question 5

In [6]:
dataset = dataset.groupby('star_rating').sample(5000)

In [7]:
def preprocess_and_vectorize3(wv):
    def f(review):
        text = str(review).lower()
        text = BeautifulSoup(text, "html.parser").get_text(strip=True)
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'[^a-z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        text = contractions.fix(text)
        stop = set(stopwords.words('english'))
        words = [lemmatizer.lemmatize(w) for w in word_tokenize(text) if w not in stop]
        vectors = [wv[w] if w in wv else [0]*300 for w in words][:50]

        if len(vectors) < 50:
            padding = np.zeros(shape=(50-len(vectors), 300))
            vectors.extend(padding)

        return vectors
    
    return f;

In [22]:
prep_input2 = lambda x: x.apply(np.array)

In [23]:
from keras.layers import Conv1D, GlobalMaxPooling1D

def test_cnn(name, feature_name, is_ternary=False):
    initializer = keras.initializers.HeNormal(seed=42)
    output_size = 3 if is_ternary else 2

    model = Sequential([
        Conv1D(filters=50, kernel_size=3, activation='relu', kernel_initializer=initializer, input_shape=(50,300)),
        Conv1D(filters=10, kernel_size=3, activation='relu', kernel_initializer=initializer),
        GlobalMaxPooling1D(),
        Dense(output_size, activation='softmax', kernel_initializer=initializer)
    ])

    model.compile(optimizer=keras.optimizers.Adam(0.001),
                loss='categorical_crossentropy' if is_ternary else 'binary_crossentropy',
                metrics=['accuracy'])
    
    model.fit(prep_input2(X_train[feature_name]), pd.get_dummies(y_train), epochs=10, batch_size=100, verbose=0)

    score = model.evaluate(prep_input2(X_test[feature_name]), pd.get_dummies(y_test), verbose=0)

    print(f"{name} accuracy: {score[1]}")

In [9]:
dataset['feature_pretrained'] = dataset['review'].apply(preprocess_and_vectorize3(wv))

In [10]:
dataset['feature_custom'] = dataset['review'].apply(preprocess_and_vectorize3(model.wv))

In [13]:
binary_ds = dataset[dataset["sentiment"] != 3]
binary_ds["sentiment"] = binary_ds["sentiment"].apply(lambda x: 1 if x==1 else 0)
X_train, X_test, y_train, y_test = train_test_split(binary_ds[["feature_pretrained", "feature_custom"]], binary_ds["sentiment"], test_size=0.2, random_state=42)

In [ ]:
prep_input2(X_train['feature_pretrained'])

: 

In [21]:
test_cnn("CNN Binary, Google", 'feature_pretrained', False)

ValueError: Invalid dtype: object